In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date

#### Widget for catalog_name

In [0]:
dbutils.widgets.text("catalog_name", "")

#### Loading data indempotently

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
table_name = "applications_info_bronze"

source_volume_path = f"/Volumes/{catalog_name}/lechster10_bronze/raw/"        # place from which AutoLoader will read raw csv files
checkpoint_path = f"/Volumes/{catalog_name}/lechster10_bronze/checkpoints/{table_name}/"              # folder where AutoLoader saves checkpoint
target_table = f"{catalog_name}.lechster10_bronze.{table_name}"
schema_path = f"/Volumes/{catalog_name}/lechster10_bronze/schemas"


# AutoLoader configuration
raw_stream_df = (spark.readStream
  .format("cloudFiles")  # enabling autoloader
  .option("cloudFiles.format", "csv")            # type of files
  .option("header", True)
  .option("cloudFiles.schemaLocation", f"{schema_path}/{table_name}/")  # place where AutoLoader automatically saves the structure of detected data schema
  .load(source_volume_path) )


# Adding new columns
processed_stream_df = (raw_stream_df
.withColumn("source_filename", col("_metadata.file_name"))
.withColumn("ingestion_timestamp", current_timestamp())
.withColumn("load_date", current_date())
)  


#### Invalid column name (space is not allowed)

In [0]:
processed_stream_df = processed_stream_df.withColumnRenamed("Content Rating", "Content_Rating")

In [0]:
processed_stream_df.columns

#### Writing data into delta table

In [0]:
# Writing into delta table

query = (processed_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)      # preventing from duplicates
    .outputMode("append")
    .trigger(availableNow=True)                         # deployed as "Batch" - processed new data and finishes
    .toTable(target_table))                  

In [0]:
# spark.sql(f" SELECT * FROM {catalog_name}.lechster10_bronze.applications_info_bronze LIMIT 5").show()

In [0]:
# spark.sql(f" SELECT count(*) FROM {catalog_name}.lechster10_bronze.applications_info_bronze").show()